In [1]:
import requests
from bs4 import BeautifulSoup
import time
from urllib.parse import urljoin
import re

BASE = "https://starwars.fandom.com"

# Примеры страниц-списков, которые могут содержать нужные ссылки
INDEX_PAGES = [
    "https://starwars.fandom.com/wiki/Category:Characters",
    "https://starwars.fandom.com/wiki/Category:Planets",
    "https://starwars.fandom.com/wiki/Category:Technology",
    "https://starwars.fandom.com/wiki/Category:Events",
    "https://starwars.fandom.com/wiki/Category:Vehicles_and_Starships",
    "https://starwars.fandom.com/wiki/Category:Artifacts",
    # можно добавить другие релевантные категории
]

def is_valid_entity_link(href: str) -> bool:
    """
    Проверка, является ли ссылка сущностью:
    - начинается с /wiki/
    - не содержит Category:, Template:, Help:, Talk:, File:, etc.
    - не ведёт на защищённые или системные страницы
    """
    if not href:
        return False
    if not href.startswith("/wiki/"):
        return False
    # исключаем системные / шаблонные / медиа страницы
    bad_prefixes = [
        "/wiki/Category:", "/wiki/Template:", "/wiki/Help:",
        "/wiki/Talk:", "/wiki/File:", "/wiki/Portal:",
        "/wiki/Special:", "/wiki/Wookieepedia:", "/wiki/MediaWiki:"
    ]
    for bp in bad_prefixes:
        if href.startswith(bp):
            return False
    # возможно, ещё фильтрация: не содержать “#” после /wiki/ (якоря) — или можно разрешить
    # исключаем страницы вида /wiki/Main_Page
    if re.match(r"^/wiki/Main_Page$", href):
        return False
    return True

def get_links_from_page(url: str) -> set[str]:
    resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    if resp.status_code != 200:
        print(f"Error fetching {url}: {resp.status_code}")
        return set()
    soup = BeautifulSoup(resp.text, "html.parser")
    links = set()
    for a in soup.find_all("a", href=True):
        href = a['href']
        if is_valid_entity_link(href):
            full = urljoin(BASE, href)
            links.add(full)
    return links

def crawl_category(category_url: str, max_pages: int = 5, delay: float = 1.0) -> set[str]:
    """
    Скрайпим одну категорию: её страницу + подстраницы (если есть навигация по страницам)
    max_pages ограничивает, сколько страниц пагинации пройти
    """
    collected = set()
    to_visit = [category_url]
    visited = set()

    pages_crawled = 0
    while to_visit and pages_crawled < max_pages:
        url = to_visit.pop(0)
        if url in visited:
            continue
        visited.add(url)
        print(f"Crawling {url}")
        links = get_links_from_page(url)
        collected.update(links)
        pages_crawled += 1

        # проверим есть ли пагинация “next page” или подстраницы категории
        soup = BeautifulSoup(requests.get(url, headers={"User-Agent": "Mozilla/5.0"}).text, "html.parser")
        # пример: искать "next page" или ссылки с class "category-page__pagination__next" или что-то подобное
        next_link = None
        pag_elems = soup.find("a", text=re.compile(r"next page", re.IGNORECASE))
        if pag_elems and 'href' in pag_elems.attrs:
            next_link = urljoin(BASE, pag_elems['href'])
        if next_link and next_link not in visited:
            to_visit.append(next_link)

        time.sleep(delay)
    return collected


if __name__ == "__main__":
    all_entity_links = set()
    for cat in INDEX_PAGES:
        links = crawl_category(cat, max_pages=10, delay=1.5)
        print(f"From category {cat}, found {len(links)} entity links")
        all_entity_links.update(links)

    # Сохранить в файл
    with open("starwars_entity_links.txt", "w", encoding="utf-8") as f:
        for link in sorted(all_entity_links):
            f.write(link + "\n")

    print(f"Total unique entity links: {len(all_entity_links)}")


Crawling https://starwars.fandom.com/wiki/Category:Characters
Error fetching https://starwars.fandom.com/wiki/Category:Characters: 404


/var/folders/x2/1s21gjpx0wbdvqp5q4_p07d80000gn/T/ipykernel_6882/1445088717.py:84: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  pag_elems = soup.find("a", text=re.compile(r"next page", re.IGNORECASE))


From category https://starwars.fandom.com/wiki/Category:Characters, found 0 entity links
Crawling https://starwars.fandom.com/wiki/Category:Planets
From category https://starwars.fandom.com/wiki/Category:Planets, found 172 entity links
Crawling https://starwars.fandom.com/wiki/Category:Technology
From category https://starwars.fandom.com/wiki/Category:Technology, found 194 entity links
Crawling https://starwars.fandom.com/wiki/Category:Events
From category https://starwars.fandom.com/wiki/Category:Events, found 61 entity links
Crawling https://starwars.fandom.com/wiki/Category:Vehicles_and_Starships
Error fetching https://starwars.fandom.com/wiki/Category:Vehicles_and_Starships: 404
From category https://starwars.fandom.com/wiki/Category:Vehicles_and_Starships, found 0 entity links
Crawling https://starwars.fandom.com/wiki/Category:Artifacts
From category https://starwars.fandom.com/wiki/Category:Artifacts, found 145 entity links
Total unique entity links: 568
